# BeeVision Full Workflow
This notebook walks through environment setup, dataset preparation, model training, and inference.
Each section now includes brief explanations and helpful plots to guide you through the full pipeline.


## Environment Setup

In [ ]:
!python --version
!pip --version

In [ ]:
!pip install --upgrade pip
!pip install -q -r ../requirements.txt

## Verify GPU and Libraries

In [ ]:
!nvidia-smi

In [ ]:
import sys, torch
print('Python', sys.version)
print('PyTorch', torch.__version__)

## Dataset Download

In [ ]:
from src.dataset import download_roboflow_dataset

# Replace placeholders with your Roboflow details
data_dir = download_roboflow_dataset(
    api_key='YOUR_API_KEY',
    workspace='YOUR_WORKSPACE',
    project='birds-and-bees',
    version=1,
)

### Inspect a Sample from the Dataset
After downloading the dataset, it's a good idea to visualize a few examples to ensure the labels look correct.

In [ ]:
from pathlib import Path
import random
import matplotlib.pyplot as plt
import matplotlib.patches as patches
from PIL import Image

# Visualize a random training image with bounding boxes
image_dir = data_dir / 'train' / 'images'
label_dir = data_dir / 'train' / 'labels'
images = list(image_dir.glob('*.jpg')) + list(image_dir.glob('*.png'))
if images:
    img_path = random.choice(images)
    label_path = label_dir / f"{img_path.stem}.txt"
    img = Image.open(img_path)
    fig, ax = plt.subplots(1, figsize=(6, 6))
    ax.imshow(img)
    if label_path.exists():
        width, height = img.size
        with open(label_path) as f:
            for line in f:
                cls, x_c, y_c, w, h = map(float, line.split())
                x = (x_c - w / 2) * width
                y = (y_c - h / 2) * height
                w_pix = w * width
                h_pix = h * height
                rect = patches.Rectangle((x, y), w_pix, h_pix, linewidth=2, edgecolor='r', facecolor='none')
                ax.add_patch(rect)
    ax.set_title(img_path.name)
    ax.axis('off')
    plt.show()
else:
    print('No images found. Ensure the dataset was downloaded correctly.')


## Model Training

In [ ]:
from src.train import train_yolov10

DATA_YAML = data_dir / 'data.yaml'
train_yolov10(data_yaml=str(DATA_YAML), model='yolov10n.pt', epochs=10)

### Review Training Metrics
The training process logs metrics to `models/train/results.csv`. Plotting these values helps monitor learning progress.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

results_csv = Path('models/train/results.csv')
if results_csv.exists():
    df = pd.read_csv(results_csv)
    cols = [c for c in ['train/box_loss', 'val/box_loss', 'metrics/mAP50(B)'] if c in df.columns]
    if cols:
        df.plot(x='epoch', y=cols)
        plt.title('Training Metrics')
        plt.xlabel('Epoch')
        plt.grid(True)
        plt.show()
    else:
        print('Expected metric columns not found in results.csv')
else:
    print('Run training to generate models/train/results.csv')


## Run Inference on an Image

In [ ]:
from src.inference import run_inference

WEIGHTS = 'models/train/weights/best.pt'  # update with your path
IMAGE = 'path/to/image.jpg'
run_inference(weights=WEIGHTS, source=IMAGE)

## Run Inference on a Video

In [ ]:
VIDEO = 'path/to/video.mp4'
run_inference(weights=WEIGHTS, source=VIDEO)